<a href="https://colab.research.google.com/github/therishiraj/Agentic-AI-workshop/blob/main/Day_4_Tooling.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Day 4 — Structured Outputs & Tool Calling
All code cells extracted from Day_4_Tooling.md (Gemini 2.5 Flash)

In [ ]:
!pip install -q groq supabase

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.7/143.7 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.4/48.4 kB 2.2 MB/s eta 0:00:00


In [ ]:
from google.colab import userdata

GROQ_API_KEY  = userdata.get("GROQ_API_KEY")
SUPABASE_URL  = userdata.get("SUPABASE_URL")
SUPABASE_KEY  = userdata.get("SUPABASE_KEY")

print("✅ Secrets loaded" if all([GROQ_API_KEY, SUPABASE_URL, SUPABASE_KEY]) else "❌ Something is missing")

✅ Secrets loaded


In [ ]:
from supabase import create_client

supabase = create_client(SUPABASE_URL, SUPABASE_KEY)

# Quick sanity check — read 3 rows
test = supabase.table("employees").select("*").limit(3).execute()
for row in test.data:
    print(row["name"], "—", row["role"])

Priya Sharma — Senior Developer
Rahul Verma — Developer
Aisha Khan — Data Scientist


In [ ]:
def get_employees(role: str = None, city: str = None) -> list[dict]:
    """Retrieve a list of employees, optionally filtered by role and/or city.

    Args:
        role: The role of the employee (e.g., 'Developer', 'HR Manager').
        city: The city where the employee is located (e.g., 'Pune', 'Bangalore').
    """
    query = supabase.table("employees").select("*")
    if role:
        query = query.eq("role", role)
    if city:
        query = query.eq("city", city)

    response = query.execute()
    return response.data

In [ ]:
import json
from groq import Groq

groq_client = Groq(api_key=GROQ_API_KEY)
MODEL = "llama-3.3-70b-versatile"   # a solid Groq model with reliable tool calling

# Maps the tool name (a string) to the real Python function
available_functions = {"get_employees": get_employees}

# Define the tools for the Groq API
tools = [
    {
        "type": "function",
        "function": {
            "name": "get_employees",
            "description": "Retrieve a list of employees, optionally filtered by role and/or city.",
            "parameters": {
                "type": "object",
                "properties": {
                    "role": {
                        "type": "string",
                        "description": "The role of the employee (e.g., 'Developer', 'HR Manager')."
                    },
                    "city": {
                        "type": "string",
                        "description": "The city where the employee is located (e.g., 'Pune', 'Bangalore')."
                    }
                }
            }
        }
    }
]

def ask_bot(question: str) -> str:
    messages = [
        {"role": "system", "content":
            "You are a helpful HR assistant. Use the provided tools to answer "
            "questions about employees. Answer in clear, friendly sentences."},
        {"role": "user", "content": question},
    ]

    # --- STEP 1: Ask Groq. Does it want to use a tool? ---
    response = groq_client.chat.completions.create(
        model=MODEL,
        messages=messages,
        tools=tools,
        tool_choice="auto",   # let the model decide
    )
    msg = response.choices[0].message

    # --- STEP 2: If it requested tool(s), run them ---
    if msg.tool_calls:
        messages.append(msg)   # record the model's request

        for call in msg.tool_calls:
            fn_name = call.function.name
            fn_args = json.loads(call.function.arguments)
            print(f"   🔧 calling {fn_name}({fn_args})")

            # Run OUR real function against the real database
            result = available_functions[fn_name](**fn_args)

            # Hand the result back to the model
            messages.append({
                "role": "tool",
                "tool_call_id": call.id,
                "name": fn_name,
                "content": json.dumps(result),
            })

        # --- STEP 3: Ask Groq again → it writes the final human answer ---
        final = groq_client.chat.completions.create(model=MODEL, messages=messages)
        return final.choices[0].message.content

    # No tool needed — the model answered directly
    return msg.content

In [ ]:
print(ask_bot("What is the salary of Aisha Khan?"))

I'm not able to find that information. I can look up employees by role or city, but I don't have access to individual employee data such as salary. Is there anything else I can help you with?


In [ ]:
print(ask_bot("What is the highest salary?"))

   🔧 calling get_employees({})
The highest salary is 1,900,000, which is received by Divya Nair, who works as an ML Engineer in the Data department in Bangalore.


## Setup

In [ ]:
!pip install google-genai pydantic

In [ ]:
import os
os.environ["GEMINI_API_KEY"] = "your-key-here"

In [ ]:
from google import genai

# The SDK automatically reads GEMINI_API_KEY from your environment.
client = genai.Client()

MODEL = "gemini-2.5-flash"   # our single model for the whole programme

## Part 1 — Structured Outputs with Pydantic

In [ ]:
from pydantic import BaseModel

# We are describing the SHAPE of a person.
# Every class that inherits from BaseModel becomes a "strict form".
class Person(BaseModel):
    name: str          # must be text
    age: int           # must be a whole number
    is_student: bool   # must be True or False

# This works - types match
p1 = Person(name="Aisha", age=21, is_student=True)
print(p1)

# This FAILS loudly - "twenty" is not an int
p2 = Person(name="Ravi", age="twenty", is_student=True)

In [ ]:
from google import genai
from google.genai import types
from pydantic import BaseModel, Field

client = genai.Client()
MODEL = "gemini-2.5-flash"

# ---------- STEP 1: Describe the shape we want ----------
class Candidate(BaseModel):
    full_name: str = Field(description="Person's full name")
    years_experience: int = Field(description="Total years of work experience")
    primary_skill: str = Field(description="Their single strongest skill")
    email: str = Field(description="Contact email address")

# ---------- STEP 2: The messy real-world input ----------
messy_text = """
Hey, I'm Priya Sharma. Been coding for about 6 years now, mostly Python
and some data science stuff. You can reach me at priya.codes@gmail.com
whenever. Cheers!
"""

# ---------- STEP 3: Ask Gemini, forcing the shape ----------
response = client.models.generate_content(
    model=MODEL,
    contents=messy_text,
    config=types.GenerateContentConfig(
        response_mime_type="application/json",   # JSON, not prose
        response_schema=Candidate,               # must match our model
    ),
)

# ---------- STEP 4: Use the result ----------
# .parsed gives you a ready-made Pydantic object - no manual JSON parsing!
candidate: Candidate = response.parsed

print("Clean, typed object:")
print("Name       :", candidate.full_name)
print("Experience :", candidate.years_experience, "years")
print("Skill      :", candidate.primary_skill)
print("Email      :", candidate.email)

## Part 2 — Handling Validation Errors Gracefully

In [ ]:
candidate = response.parsed        # if anything went wrong, this can be None or raise
print(candidate.full_name)         # crashes if candidate is None

In [ ]:
from pydantic import BaseModel, Field, field_validator

class Candidate(BaseModel):
    full_name: str
    years_experience: int
    primary_skill: str
    email: str

    @field_validator("years_experience")
    @classmethod
    def years_must_be_sane(cls, v: int) -> int:
        if v < 0 or v > 60:
            raise ValueError("years_experience must be between 0 and 60")
        return v

In [ ]:
from pydantic import ValidationError

def extract_candidate(text: str, max_retries: int = 2) -> Candidate | None:
    for attempt in range(1, max_retries + 1):
        print(f"Attempt {attempt}...")
        try:
            response = client.models.generate_content(
                model=MODEL,
                contents=text,
                config=types.GenerateContentConfig(
                    response_mime_type="application/json",
                    response_schema=Candidate,
                ),
            )
            # .parsed runs Pydantic validation for us. If a rule fails,
            # it raises ValidationError, which we catch below.
            return response.parsed

        except ValidationError as e:
            print(f"Validation failed: {e}")
            # Loop runs again -> Gemini gets another chance
        except Exception as e:
            print(f"API/parse error: {e}")

    print("Gave up after retries.")
    return None

result = extract_candidate(messy_text)
if result:
    print("Got clean data:", result.full_name)
else:
    print("Could not extract - handle this case in your app.")

## Part 3 — Tool Calling Fundamentals

In [ ]:
def get_weather(city: str) -> str:
    """Get the current weather for a given city.

    Args:
        city: The name of the city, e.g. "Delhi".
    """
    fake_db = {
        "delhi": "34°C, sunny",
        "london": "12°C, rainy",
        "tokyo": "19°C, cloudy",
    }
    return fake_db.get(city.lower(), "No data for that city")

def multiply(a: float, b: float) -> float:
    """Multiply two numbers together accurately.

    Args:
        a: The first number.
        b: The second number.
    """
    return a * b

In [ ]:
from google.genai import types

response = client.models.generate_content(
    model=MODEL,
    contents="What's the weather in Tokyo?",
    config=types.GenerateContentConfig(
        tools=[get_weather, multiply],   # just pass the functions!
    ),
)

print(response.text)
# The SDK automatically:
#   1. Sent your question + the tool menu to Gemini
#   2. Saw Gemini ask for get_weather(city="Tokyo")
#   3. Ran YOUR function, got "19°C, cloudy"
#   4. Sent the result back, got the final sentence
# Output -> "The weather in Tokyo is currently 19°C and cloudy."

response2 = client.models.generate_content(
    model=MODEL,
    contents="What is 847 times 219?",
    config=types.GenerateContentConfig(tools=[get_weather, multiply]),
)
print(response2.text)
# Output -> "847 times 219 equals 185,493."

In [ ]:
from google.genai import types

config = types.GenerateContentConfig(
    tools=[get_weather, multiply],
    # Turn OFF auto-calling so WE handle the tool call manually
    automatic_function_calling=types.AutomaticFunctionCallingConfig(disable=True),
)

response = client.models.generate_content(
    model=MODEL,
    contents="What's the weather in Delhi?",
    config=config,
)

# Did Gemini ask for a tool?
part = response.candidates[0].content.parts[0]
if part.function_call:
    call = part.function_call
    print(f"Gemini wants to call: {call.name}({dict(call.args)})")
    # -> Gemini wants to call: get_weather({'city': 'Delhi'})

    # WE decide to run it (safety!) and could feed the result back
    available = {"get_weather": get_weather, "multiply": multiply}
    result = available[call.name](**call.args)
    print("   result:", result)

## Part 4 — Multi-Tool Routing & Ambiguity

In [ ]:
def convert_currency(amount: float, from_cur: str, to_cur: str) -> str:
    """Convert an amount of money from one currency to another.

    Args:
        amount: How much money to convert.
        from_cur: The 3-letter source currency code, e.g. "USD".
        to_cur: The 3-letter target currency code, e.g. "INR".
    """
    rates = {"USD_INR": 83.0, "INR_USD": 1 / 83.0, "EUR_INR": 90.0}
    key = f"{from_cur.upper()}_{to_cur.upper()}"
    if key not in rates:
        return "Rate not available"
    return f"{amount * rates[key]:.2f} {to_cur.upper()}"

# Hand ALL THREE tools to Gemini - it picks the right one every time
tools = [get_weather, multiply, convert_currency]

for question in [
    "What's the weather in Delhi?",
    "What is 50 times 12?",
    "Convert 100 US dollars to Indian rupees.",
]:
    resp = client.models.generate_content(
        model=MODEL,
        contents=question,
        config=types.GenerateContentConfig(tools=tools),
    )
    print(f"Q: {question}\nA: {resp.text}\n")

In [ ]:
config = types.GenerateContentConfig(
    tools=[convert_currency],
    system_instruction=(
        "You are a helpful assistant. If a request is missing information "
        "needed to call a tool, ASK the user a clarifying question instead "
        "of guessing."
    ),
)

resp = client.models.generate_content(
    model=MODEL,
    contents="Convert it to rupees for me.",
    config=config,
)
print(resp.text)
# Likely: "Sure! How much would you like to convert, and from which currency?"

In [ ]:
def lookup(x):          # no type hints, no docstring
    ...

In [ ]:
def lookup_order_status(order_id: str) -> str:
    """Look up the delivery status of a customer order using their order ID.

    Use this whenever a user asks 'where is my order' or mentions an
    order number.

    Args:
        order_id: The customer's order ID, e.g. "ORD-4821".
    """
    ...

## Part 5 — Demo A: The Invoice Extractor

In [ ]:
from pydantic import BaseModel, Field

class Invoice(BaseModel):
    invoice_number: str = Field(description="The invoice number/ID")
    vendor: str = Field(description="Company that issued the invoice")
    total_amount: float = Field(description="Total payable amount as a number")
    due_date: str = Field(description="Due date in YYYY-MM-DD format")

email = """
Hi team, please process payment for invoice INV-2026-0091 from
BrightWare Solutions. The total comes to 45,600 rupees and it's
due by the 15th of August 2026. Thanks!
"""

response = client.models.generate_content(
    model=MODEL,
    contents=email,
    config=types.GenerateContentConfig(
        response_mime_type="application/json",
        response_schema=Invoice,
    ),
)

invoice: Invoice = response.parsed
print(invoice)
# invoice_number='INV-2026-0091' vendor='BrightWare Solutions'
# total_amount=45600.0 due_date='2026-08-15'

## Quick Reference Card

In [ ]:
from google import genai
from google.genai import types
from pydantic import BaseModel

client = genai.Client()
MODEL = "gemini-2.5-flash"

# STRUCTURED OUTPUT - the recipe
class MyModel(BaseModel):
    field_a: str
    field_b: int

resp = client.models.generate_content(
    model=MODEL,
    contents="...your text...",
    config=types.GenerateContentConfig(
        response_mime_type="application/json",
        response_schema=MyModel))
obj = resp.parsed          # ready-made typed object

# TOOL CALLING - the easy way
def my_tool(x: str) -> str:
    """Clear docstring telling Gemini exactly what this does."""
    return "..."

resp = client.models.generate_content(
    model=MODEL,
    contents="...user question...",
    config=types.GenerateContentConfig(tools=[my_tool]))
print(resp.text)           # Gemini called the tool for you